# 4주차 · 반도체 공정 불량 예측 🏭

마지막 주 미니 대회는 **실제 반도체 공장 데이터**입니다! 웨이퍼(반도체 원판) 하나를 만들 때 측정된 **센서 값 590개**를 보고, 그 웨이퍼가 **불량(1)인지 정상(0)인지** 예측해요.

이 데이터엔 현실적인 어려움이 두 가지 있어요:
1. **불균형**: 불량이 전체의 6.6%뿐이라 "전부 정상!"으로 찍으면 F1이 낮아요. (F1은 불량을 실제로 잡아내야 점수를 주는 지표)
2. **결측치**: 센서가 가끔 측정에 실패해서 빈 값이 많아요 → 채워 넣는 **전처리**가 필요합니다.

**진행 방법 (꼭 읽어주세요!)**
1. 셀을 **위에서부터 하나씩** `Shift+Enter`로 실행하세요.
2. **✏️ 표시된 셀만** 바꿔보면 됩니다.
3. 마지막 셀이 `submission.csv`를 다운로드해줘요 → 웹 **Submit 탭**에 업로드.

- 평가 지표: **F1** (높을수록 좋음)
- 기본 코드 그대로도 베이스라인('전부 정상' 찍기)은 이깁니다. 마지막 주도 화이팅! 💪

## 1. 데이터 불러오기  *(그냥 실행하세요)*

센서 데이터를 읽어옵니다. `id`와 정답(`target`)을 뺀 나머지 590개 컬럼이 전부 센서 측정값이에요.

실행하면 데이터 크기와 함께 `불량 비율: 0.066` 정도가 찍혀요 — 100개 중 겨우 6~7개가 불량이라는 뜻입니다.

In [ ]:
import pandas as pd  # 표 데이터를 다루는 라이브러리

# 데이터 공개 링크 (운영진이 채워둔 것 — 수정하지 마세요!)
TRAIN_URL = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/manufacturing/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/manufacturing/test.csv"

if TRAIN_URL:
    train = pd.read_csv(TRAIN_URL); test = pd.read_csv(TEST_URL)  # URL에서 바로 읽기
else:
    # (예비용) URL이 없을 때만 직접 업로드 — 보통 실행되지 않아요
    from google.colab import files; files.upload()
    train = pd.read_csv("train.csv"); test = pd.read_csv("test.csv")

# id와 정답(target)을 뺀 나머지 = 센서 컬럼 590개
sensor_cols = [c for c in train.columns if c not in ("id", "target")]

print("train:", train.shape, "| test:", test.shape)
print("불량 비율:", round(train["target"].mean(), 3))  # 0.066 근처 = 심한 불균형!

## 2. ✏️ 여기를 바꿔보세요 — 전처리 & 모델 학습

여기서는 세 단계를 **파이프라인**으로 묶어 한 번에 처리합니다 (순서대로 자동 실행돼요):
1. `SimpleImputer(strategy="median")` — 비어있는 센서 값을 그 센서의 **중앙값**으로 채우기 (모델은 빈 값을 못 다뤄요)
2. `StandardScaler` — 센서마다 단위가 제각각이라, 전부 **평균 0·표준편차 1**로 통일 (스케일링)
3. `LogisticRegression(class_weight="balanced")` — 수가 적은 불량(6.6%)을 틀리면 벌점을 크게 줘서 불량을 적극적으로 잡게 함

실행하면 몇십 초 안에 학습이 끝나고 파이프라인 정보가 출력됩니다. 이 기본 예시로 '전부 정상' 찍기(F1 0.48)는 넘어요.

**점수(F1) 올리는 실험 아이디어** — 하나씩 바꿔서 제출해보세요:
- **⚠️ 예측 임계값이 이 대회의 핵심이에요.** 불량이 6.6%뿐이라, 기본 기준(확률 0.5 이상만 불량)으로는 불량을 거의 못 잡습니다. 아래 3번 셀 주석을 풀어 기준을 0.3~0.4로 낮춰보세요 — F1이 크게 오릅니다.
- 모델 교체: `from sklearn.ensemble import RandomForestClassifier` 후 `RandomForestClassifier(n_estimators=300, class_weight="balanced")`. **단, 트리 모델은 확률을 보수적으로 줘서 기본 기준 0.5로는 F1이 0에 가깝게 나와요.** 반드시 위의 임계값 낮추기와 함께 쓰세요. (직접 확인해보면 임계값이 왜 중요한지 몸으로 알게 됩니다.)
- 결측치 채우기 방법: `strategy="median"` → `"mean"` 또는 `"most_frequent"`
- 변화 없는 센서 제거: `from sklearn.feature_selection import VarianceThreshold` 를 파이프라인에 추가
- **로컬 검증**: 제출 전에 train을 나눠(StratifiedKFold) F1을 미리 재보면, 어떤 임계값이 좋은지 제출 없이 찾을 수 있어요.

In [ ]:
from sklearn.pipeline import make_pipeline        # 여러 단계를 한 줄로 묶는 도구
from sklearn.impute import SimpleImputer           # 결측치(빈 값) 채우기
from sklearn.preprocessing import StandardScaler   # 스케일 통일
from sklearn.linear_model import LogisticRegression  # 기본 분류 모델

# ✏️ 여기를 바꿔보세요! 모델을 RandomForestClassifier로 바꾸거나, strategy를 바꿔 실험
clf = make_pipeline(
    SimpleImputer(strategy="median"),   # 1) 빈 센서 값을 중앙값으로 채움
    StandardScaler(),                   # 2) 모든 센서를 같은 스케일로 통일
    LogisticRegression(max_iter=2000, class_weight="balanced"),  # 3) 불균형 대응 분류 모델
)

# 학습: 센서 590개 → 불량 여부의 패턴을 찾습니다 (몇십 초 정도)
clf.fit(train[sensor_cols], train["target"])

## 3. 예측 & 제출 파일 저장  *(실행하면 자동 다운로드)*

학습된 파이프라인으로 test 웨이퍼들의 불량 여부를 예측하고 `submission.csv`로 저장합니다. `예측된 불량 수`가 0이 아닌 두 자릿수 정도로 찍히면 정상이에요 (0이면 모델이 불량을 하나도 못 잡는 것!).

💡 **임계값 실험**: 기본 `predict`는 불량 확률이 50%를 넘어야 불량이라고 해요. 아래 주석의 코드로 기준을 35% 등으로 낮추면 불량을 더 많이 잡아 F1이 오를 수 있어요.

👉 다운로드된 `submission.csv`를 웹사이트의 **Submit 탭**에 업로드하세요. 여러 번 제출 가능!

In [ ]:
# 기본 예측: 불량 확률이 0.5(50%)를 넘으면 1(불량)로 판정
test["prediction"] = clf.predict(test[sensor_cols])

# ✏️ 임계값을 바꿔보려면 위 줄을 지우고(#로 주석 처리) 아래 두 줄의 #을 지워보세요:
# proba = clf.predict_proba(test[sensor_cols])[:, 1]      # 각 웨이퍼의 '불량일 확률'
# test["prediction"] = (proba > 0.35).astype(int)          # 기준을 0.35로 낮춰 더 과감하게 잡기

# 제출 규칙: id, prediction 두 컬럼만 저장 (index=False: 행 번호 제외)
test[["id", "prediction"]].to_csv("submission.csv", index=False)
print("저장 완료 · 예측된 불량 수:", int(test["prediction"].sum()))

try:
    # 코랩에서 실행 중이면 자동 다운로드
    from google.colab import files; files.download("submission.csv")
except Exception:
    print("왼쪽 파일탭에서 submission.csv를 내려받으세요.")

## 🆘 막히면 여기를 보세요

| 증상 | 해결 |
|---|---|
| `NameError: name 'train' is not defined` | 위쪽 셀을 건너뛰었어요. 맨 위부터 순서대로 실행하세요. |
| `Input contains NaN` 에러 | 파이프라인의 `SimpleImputer`가 빠진 거예요 — 결측치 채우기는 꼭 필요해요. |
| 수렴 경고(ConvergenceWarning) | 빨간 에러가 아니면 무시해도 됩니다. `max_iter`를 키우면 사라져요. |
| 예측된 불량 수가 0 | 모델이 불량을 하나도 못 잡는 상태 — `class_weight="balanced"`가 빠졌는지 확인하거나 임계값을 낮춰보세요. |
| 데이터 로드 URL 에러 | 인터넷 연결 확인 후 셀을 한 번 더 실행. |

4주 완주 축하해요! 여러분은 이제 시계열·이미지·텍스트·표 데이터를 모두 다뤄본 사람입니다. 🎓✨

## 더 나아가기 (선택)

점수는 파라미터 숫자보다 **"데이터를 어떻게 보여주느냐"**(피처 엔지니어링·검증 전략·앙상블)에서 갈립니다. 아래는 정답 코드가 아니라 **방향과 시작점 몇 줄**만 적어둔 것이에요.

> AI(ChatGPT·Claude·Gemini)에게 물어보며 해도 좋고, 코랩을 자유롭게 고치거나 새 셀을 만들어도 됩니다. **제출 형식(id, prediction)만 맞으면 무엇을 하든 괜찮아요.**

기본 흐름으로 이미 베이스라인은 넘었어요. 여기서부터는 실력 차가 나는 구간입니다. 아래 갈래들을 골라 실험해보세요 (셀은 주석 처리돼 있으니 `#`을 지워 쓰면 됩니다).

- **임계값 최적화 (이 대회의 핵심)**: `predict_proba`로 확률을 뽑은 뒤, F1이 가장 높아지는 기준값을 직접 찾기
- **로컬 검증**: `StratifiedKFold`로 제출 전에 F1을 미리 재보기 (불량 비율을 유지한 채 나눠야 함)
- **피처 정리**: 값이 거의 안 변하는 센서·결측이 너무 많은 센서 제거, 상관 높은 센서 정리, `SelectKBest`·변수 중요도로 선택
- **불균형 대응**: `class_weight`, 또는 소수 클래스를 늘리는 SMOTE
- **더 강한 모델 & 앙상블**: LightGBM·XGBoost·GradientBoosting, 여러 모델 확률을 평균내는 soft voting


In [ ]:
# ── 로컬 CV로 최적 임계값 찾기 (제출 없이 F1 미리 재보기) ──
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import f1_score
# import numpy as np
#
# X = train[sensor_cols]; y = train["target"]
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# oof = np.zeros(len(y))          # out-of-fold 예측 확률을 모아둘 배열
# for tr, va in skf.split(X, y):
#     clf.fit(X.iloc[tr], y.iloc[tr])
#     oof[va] = clf.predict_proba(X.iloc[va])[:, 1]
# for t in np.arange(0.1, 0.6, 0.05):   # 여러 임계값을 훑어 F1 비교
#     print(round(t, 2), round(f1_score(y, (oof > t).astype(int)), 4))
# # 여기서 가장 좋았던 값을 제출 셀의 (proba > 0.35) 자리에 넣어보세요


In [ ]:
# ── 쓸모없는 센서 정리 + 중요한 센서만 고르기 ──
# from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
# 결측이 너무 많은 센서는 후보에서 빼기 (예: 절반 이상 비어있으면 제거)
# keep = [c for c in sensor_cols if train[c].isna().mean() < 0.5]
# 파이프라인 맨 앞에 VarianceThreshold()를 넣으면 거의 변하지 않는 센서가 자동 제거됩니다.
# SelectKBest(f_classif, k=100) 으로 target과 관련 높은 상위 k개만 남길 수도 있어요.
# (make_pipeline 안에 SimpleImputer 다음, 모델 앞 단계로 끼워 넣으면 됩니다.)


In [ ]:
# ── 더 강한 모델 & 앙상블 ──
# 코랩엔 아래 라이브러리가 대부분 있어요 (없으면 !pip install lightgbm xgboost)
# from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
# from lightgbm import LGBMClassifier
# from xgboost import XGBClassifier
# 트리 계열은 스케일링이 필요 없지만, 확률을 보수적으로 줘서 임계값 조정이 특히 중요해요.
# 불균형 대응: LGBMClassifier(class_weight="balanced") 또는 scale_pos_weight 사용
# 여러 모델의 확률을 평균내는 soft voting:
# ens = VotingClassifier([("lr", clf), ("lgb", LGBMClassifier())], voting="soft")
# 소수 클래스를 늘리는 SMOTE도 한 갈래: from imblearn.over_sampling import SMOTE
# 앙상블한 확률에도 위의 임계값 탐색을 똑같이 적용하세요.
